# Printing and Plotting Results

Contains two sections:
1. How to create a summary table.
2. How to create plots showing the ranks of the similarity measures.

## Summary Table

This section of the notebook, creates the overview table in our paper. This code can be easily adjusted to also output more detailed tables.

#### This notebook summarizes a specific domain and dataset compared to the tables_and_plots notebook

In [58]:
import re
from pathlib import Path

import pandas as pd
import pandas.io.formats.style
import os

import matplotlib.pyplot as plt

from repsim.benchmark.paths import BASE_PATH


measure_to_abbrv = {
    "AlignedCosineSimilarity": "AlignCos",
    "CKA": "CKA",
    "ConcentricityDifference": "ConcDiff",
    "DistanceCorrelation": "DistCorr",
    "EigenspaceOverlapScore": "EOS",
    "GeometryScore": "GS",
    "Gulp": "GULP",
    "HardCorrelationMatch": "HardCorr",
    "IMDScore": "IMD",
    "JaccardSimilarity": "Jaccard",
    "LinearRegression": "LinReg",
    "MagnitudeDifference": "MagDiff",
    "OrthogonalAngularShapeMetricCentered": "AngShape",
    "OrthogonalProcrustesCenteredAndNormalized": "OrthProc",
    "PWCCA": "PWCCA",
    "PermutationProcrustes": "PermProc",
    "ProcrustesSizeAndShapeDistance": "ProcDist",
    "RSA": "RSA",
    "RSMNormDifference": "RSMDiff",
    "RankSimilarity": "RankSim",
    "SVCCA": "SVCCA",
    "SecondOrderCosineSimilarity": "2nd-Cos",
    "SoftCorrelationMatch": "SoftCorr",
    "UniformityDifference": "UnifDiff",
    "RTD": "RTD",
    # my metrics 
    "CKA_RBF": "CKA_RBF",
    "CKA_RBF_AUC": "CKA_RBF_AUC",
    "RWKA": "RWKA",
    "RWKA_AUC": "RWKA_AUC",
}

measure_types = [
    ("AlignCos", "Alignment"),
    ("HardCorr", "Alignment"),
    ("AngShape", "Alignment"),
    ("LinReg", "Alignment"),
    ("OrthProc", "Alignment"),
    ("PermProc", "Alignment"),
    ("ProcDist", "Alignment"),
    ("SoftCorr", "Alignment"),

    ("EOS", "RSM"),
    ("CKA", "RSM"),
    ("DistCorr", "RSM"),
    ("GULP", "RSM"),
    ("RSA", "RSM"),
    ("RSMDiff", "RSM"),

    ("MagDiff", "Statistic"),
    ("ConcDiff", "Statistic"),
    ("UnifDiff", "Statistic"),

    ("GS", "Topology"),
    ("IMD", "Topology"),
    ("RTD", "Topology"),

    ("Jaccard", "Neighbors"),
    ("RankSim", "Neighbors"),
    ("2nd-Cos", "Neighbors"),

    ("PWCCA", "CCA"),
    ("SVCCA", "CCA"),

    # my metrics
    ("CKA_RBF", "Manifold"),
    ("CKA_RBF_AUC", "Manifold"),
    ("RWKA", "Manifold"),
    ("RWKA_AUC", "Manifold"),

]

measure_type_order = ["CCA", "Alignment", "RSM", "Neighbors", "Topology", "Statistic", "Manifold"]


In [59]:
target_domain = 'graphs'
target_dataset = 'cora'

Step 1: Load all results.

In [60]:
if target_domain == 'nlp':
    cleaned_dfs = []
    nlp_root = BASE_PATH / "results"
    # results_root = os.path.join(BASE_PATH, "results", "graphs", "new_metrics_demo")
    # results_root = BASE_PATH / "results" / "graphs" / "new_metrics_demo"
    for path in nlp_root.glob("*.csv"):
        df = pd.read_csv(path, index_col=0)
        setting = path.name.split("_")[0]

        pattern = r'(?<=_)sst2(?=_)|(?<=_)mnli(?=_)'
        match = re.search(pattern, path.name)
        if match is None:
            continue
        dataset = match.group(0)
        
        if dataset != target_dataset:
            continue

        token = path.name.split("_")[-1].split(".")[0]

        if "smollm" in path.name:
            # not true, but we want to group standard non-aggregated token results for the llm with the cls token results for bert and albert
            token = "cls"

        df["Token"] = token
        df["Setting"] = setting
        df["Dataset"] = dataset
        cleaned_dfs.append(df)

    data = pd.concat(cleaned_dfs).reset_index(drop=True)
    nlp_data = data
    print("loaded NLP data")
else:
    print(f"loading {target_domain} data not NLP data")


loading graphs data not NLP data


In [61]:
if target_domain == 'graphs':
    cleaned_dfs = []
    # root = BASE_PATH /"paper_results" /"graphs"
    root = BASE_PATH / "results"
    for path in root.glob("*.csv"):
        if path.name.endswith("backup.csv"):
            continue

        df = pd.read_csv(path, index_col=0)
        pattern = r"augmentation|label_test|layer_test|output_correlation|shortcut"
        match = re.search(pattern, path.name)
        if match is None:
            continue
        pattern_to_setting = {
            "augmentation": "aug",

            "label_test": "mem",
            "layer_test": "mono",
            "output_correlation": "correlation",
            "shortcut": "sc",
        }
        setting = pattern_to_setting[match.group(0)]

        pattern = r"(?<=_)cora(?=_)|(?<=_)flickr(?=_)|(?<=_)ogbn-arxiv(?=_)"
        match = re.search(pattern, path.name)
        if match is None or match.group(0) != target_dataset:
            continue
        dataset = match.group(0)
        print(f"loading {setting} setting for {dataset} dataset")

        df["Setting"] = setting
        
        df["Dataset"] = dataset
        cleaned_dfs.append(df)

    data = pd.concat(cleaned_dfs).reset_index(drop=True)
    graph_data = data
    print("loaded graph data")
else:
    print(f"loading {target_domain} data not graph data")

loading mem setting for cora dataset
loading correlation setting for cora dataset
loading aug setting for cora dataset
loading aug setting for cora dataset
loading sc setting for cora dataset
loaded graph data


In [62]:
if target_domain == 'vision':
    cleaned_dfs = []
    root = BASE_PATH /"paper_results" /"vision_cameraready"
    for path in root.glob("*.csv"):
        df = pd.read_csv(path, index_col=0)
        pattern = r"aug|augment|mem|randomlabel|mono|correlation|output|sc|shortcut"
        match = re.search(pattern, path.name)
        pattern_to_setting = {
            "aug": "aug",
            "augment": "aug",
            "mem": "mem",
            "randomlabel": "mem",
            "mono": "mono",
            "correlation": "correlation",
            "output": "correlation",
            "sc": "sc",
            "shortcut": "sc",
        }
        setting = pattern_to_setting[match.group(0)]

        pattern = r"(?<=_)in100(?=_)|(?<=_)c100(?=_)|in100(?=_)|c100(?=_)|C100(?=_)"
        match = re.search(pattern, path.name)
        if match is None or match.group(0) != target_dataset:
            continue
        dataset = match.group(0)
        if dataset == "C100":
            dataset = "c100"

        df["Setting"] = setting
        df["Dataset"] = dataset
        cleaned_dfs.append(df)

    data = pd.concat(cleaned_dfs).reset_index(drop=True)
    vision_data = data
    print("loaded vision data")
else:
    print(f"loading {target_domain} data not vision data")

loading graphs data not vision data


In [63]:
data["architecture"].value_counts()

architecture
GCN          464
GraphSAGE    464
GAT          464
PGNN         464
Name: count, dtype: int64

## Clean and Pivot

Step 2: Combine data into a big dataframe, clean up column names etc., and select data to be shown in table.

In [64]:
# ----------------------------------------------------------------------------------------------------------------------
# Combine data
# ----------------------------------------------------------------------------------------------------------------------
# data = pd.concat([nlp_data, graph_data, vision_data])

print(data.columns)

data = data.rename(
    columns={
        "functional_similarity_measure": "Functional Similarity Measure",
        "similarity_measure": "Representational Similarity Measure",
        "quality_measure": "Quality Measure",
    }
)

idx = data.Setting == "correlation"
data.loc[idx, "value"] = data.loc[idx, "corr"]

idx = (data.Setting == "correlation") & (data["Functional Similarity Measure"] == "AbsoluteAccDiff")
data.loc[idx, "Setting"] = "acc_corr"

# ----------------------------------------------------------------------------------------------------------------------
# Exclude data not to be shown in table.
# ----------------------------------------------------------------------------------------------------------------------
# idx = (data.Setting == "correlation") & (data["Functional Similarity Measure"] != "JSD")
# data = data.loc[~idx]

# idx = (data.Setting.isin(["aug", "mem", "sc"])) & (data["Quality Measure"] != "AUPRC")
# data = data.loc[~idx]

# idx = (data.Setting.isin(["correlation", "acc_corr"])) & (data["Quality Measure"] != "spearmanr")
# data = data.loc[~idx]

# # idx = (data.Setting.isin(["mono"])) & (data["Quality Measure"] != "violation_rate")
# idx = (data.Setting.isin(["mono"])) & (data["Quality Measure"] != "correlation")
# data = data.loc[~idx]

if target_domain == 'nlp':
    idx = (data.Token.isin(["mean"]))
    data = data.loc[~idx]


# ----------------------------------------------------------------------------------------------------------------------
# Clean up names etc.
# ----------------------------------------------------------------------------------------------------------------------


def beautify_df(data):
    data.loc[:, "Representational Similarity Measure"] = data["Representational Similarity Measure"].map(
        measure_to_abbrv
    )
    data.loc[:, "architecture"] = data["architecture"].map(
        {
            "smollm2-1.7b": "SmolLM2",
            "albert-base-v2": "ALBERT",
            "BERT-L": "BERT",
            "GCN": "GCN",
            "GAT": "GAT",
            "GraphSAGE": "SAGE",
            "VGG11": "VGG11",
            "VGG19": "VGG19",
            "ResNet18": "RNet18",
            "ResNet34": "RNet34",
            "ResNet101": "RNet101",
            "ViT_B32": "ViT_B32",
            "ViT_L32": "ViT_L32",
            "PGNN": "P-GNN",
        }
    )
    data.loc[:, "domain"] = data["domain"].map({"NLP": "Text", "GRAPHS": "Graph", "VISION": "Vision"})
    data.loc[:, "Dataset"] = data["Dataset"].map(
        {
            "mnli_aug_rate0": "MNLI",
            "mnli_mem_rate0": "MNLI",
            "mnli": "MNLI",
            "sst2_sc_rate0558": "SST2",
            "sst2_mem_rate0": "SST2",
            "sst2_sft": "SST2",
            "sst2_sft_sc_rate0558": "SST2",
            "mnli_sc_rate0354": "MNLI",
            "sst2_aug_rate0": "SST2",
            "sst2": "SST2",
            "flickr": "flickr",
            "ogbn-arxiv": "arXiv",
            "cora": "Cora",
            "in100": "IN100",
            "c100": "CIFAR100",
        }
    )
    data.loc[:, "Setting"] = data["Setting"].map(
        {
            "aug": "Augmentation",
            "mem": "Random Labels",
            "correlation": "JSD Corr.",
            "acc_corr": "Acc Corr.",
            "mono": "Layer Mono.",
            "sc": "Shortcuts",
        }
    )
    column_order = ["Acc Corr.", "JSD Corr.", "Random Labels", "Shortcuts", "Augmentation", "Layer Mono."]
    data.loc[:, "Setting"] = pd.Categorical(
        data["Setting"],
        categories=column_order,
        ordered=True,
    )
    data.loc[:, "Quality Measure"] = data["Quality Measure"].map(
        {"violation_rate": "Conformity Rate", "AUPRC": "AUPRC", "spearmanr": "Spearman", "correlation": "Spearman"}
    )
    data.loc[data["Quality Measure"] == "Conformity Rate", "value"] = (
        1 - data.loc[data["Quality Measure"] == "Conformity Rate", "value"]
    )  # must be run in conjunction with the above renaming

    data = data.rename(
        columns={
            "domain": "Domain",
            "architecture": "Arch.",
            "Representational Similarity Measure": "Sim Meas.",
            "Quality Measure": "Eval.",
            "Setting": "Test",
        }
    )
    data = pd.merge(data, pd.DataFrame.from_records(measure_types, columns=["Sim Meas.", "Measure Type"]), how="left", on="Sim Meas.")
    data.loc[:, "Measure Type"] = pd.Categorical(data["Measure Type"], categories=measure_type_order, ordered=True)
    data.loc[data.Test.isin(["Acc Corr.", "JSD Corr."]), "Type"] = "Grounding by Prediction"
    data.loc[data.Test.isin(["Random Labels", "Shortcuts", "Augmentation", "Layer Mono."]), "Type"] = (
        "Grounding by Design"
    )
    return data, column_order


data, column_order = beautify_df(data)

# ----------------------------------------------------------------------------------------------------------------------
# Create aggregated overview table
# ----------------------------------------------------------------------------------------------------------------------
# idx = data["Dataset"].isin(["MNLI", "flickr", "IN100"]) & data["Arch."].isin(["SAGE", "BERT", "RNet18"])
# idx = data["Dataset"].isin(["SST2", "flickr", "IN100"]) & data["Arch."].isin(["SAGE", "BERT", "RNet18"])
# idx = data["Dataset"].isin(["SST2", "cora", "flickr", "IN100"]) & data["Arch."].isin(["GCN", "GraphSAGE", "GAT", "SAGE", "BERT", "RNet18"])


# pivot = pd.pivot_table(
#     data.loc[idx],
#     index=["Measure Type", "Sim Meas."],  # <---
#     # index="Sim Meas.",
#     columns=["Type", "Test", "Eval.", "Domain", "Dataset", "Arch."],
#     values="value",
# )
pivot = pd.pivot_table(
    data,
    index=["Measure Type", "Sim Meas."],  # <---
    # index="Sim Meas.",
    columns=["Type", "Test", "Eval.", "Domain", "Dataset", "Arch."],
    values="value",
)
pivot = pivot.sort_values(by=["Measure Type", "Sim Meas."], axis="index")  # <---
# pivot = pivot.sort_values(by="Sim Meas.", axis="index")
pivot = pivot.reindex(measure_type_order, axis="index", level=0)  # <---
pivot = pivot.reindex(column_order, axis="columns", level="Test")
pivot = pivot.reindex(["Grounding by Prediction", "Grounding by Design"], axis="columns", level="Type")
pivot

Index(['similarity_measure', 'quality_measure', 'value', 'domain',
       'architecture', 'representation_dataset', 'identifier', 'Setting',
       'Dataset', 'functional_similarity_measure', 'corr', 'pval'],
      dtype='object')


Type                     Grounding by Prediction                      \
Test                                   Acc Corr.                       
Eval.                                   Spearman                       
Domain                                     Graph                       
Dataset                                     Cora                       
Arch.                                        GAT       GCN      SAGE   
Measure Type Sim Meas.                                                 
CCA          PWCCA                     -0.264699 -0.194802  0.062835   
             SVCCA                     -0.161198  0.071516 -0.123879   
Alignment    AlignCos                  -0.320348  0.066634  0.125132   
             AngShape                  -0.129078  0.072571 -0.294304   
             HardCorr                  -0.141503  0.051328 -0.108188   
             LinReg                    -0.129318  0.051186 -0.215562   
             OrthProc                  -0.129078  0.072571 -0.294304   
             PermProc                  -0.279900  0.110902  0.185588   
             ProcDist                  -0.209974  0.084381  0.010812   
             SoftCorr                   0.018241  0.187564 -0.053270   
RSM          CKA                        0.030733  0.076992 -0.175435   
             DistCorr                   0.028948  0.058387 -0.178270   
             EOS                       -0.046793 -0.215141  0.073180   
             GULP                      -0.122006 -0.223586  0.071730   
             RSA                       -0.307328  0.149959  0.044963   
             RSMDiff                   -0.136348 -0.048293  0.076015   
Neighbors    2nd-Cos                   -0.116190  0.114135  0.036788   
             Jaccard                   -0.147980  0.160185 -0.110957   
             RankSim                    0.358880  0.099555 -0.123418   
Topology     IMD                       -0.006411 -0.081214  0.011999   
Statistic    ConcDiff                  -0.222400  0.184463 -0.251451   
             MagDiff                   -0.214270  0.148639 -0.131263   
             UnifDiff                  -0.075081 -0.100478 -0.045095   
Manifold     CKA_RBF                    0.085127  0.067293 -0.162975   
             CKA_RBF_AUC                0.018241  0.064457 -0.172337   
             RWKA                      -0.033971 -0.017417 -0.155723   
             RWKA_AUC                  -0.207595  0.074881  0.012658   

Type                                                   Grounding by Design  \
Test                     JSD Corr.                           Random Labels   
Eval.                     Spearman                                   AUPRC   
Domain                       Graph                                   Graph   
Dataset                       Cora                                    Cora   
Arch.                          GAT       GCN      SAGE                 GAT   
Measure Type Sim Meas.                                                       
CCA          PWCCA       -0.061595 -0.206405  0.313027            0.277280   
             SVCCA        0.213828  0.565466  0.031474            0.421563   
Alignment    AlignCos     0.061632  0.341500  0.327154            0.432640   
             AngShape     0.167915  0.680556  0.213964            0.424196   
             HardCorr     0.388486  0.674724  0.133080            0.423522   
             LinReg       0.064150  0.352781  0.210497            0.456432   
             OrthProc     0.167915  0.680556  0.213964            0.424196   
             PermProc    -0.002581  0.666463  0.211828            0.435082   
             ProcDist    -0.064327  0.646292  0.161000            0.433137   
             SoftCorr     0.142101  0.747970  0.021894            0.423338   
RSM          CKA          0.489973  0.672584 -0.015019            0.423660   
             DistCorr     0.528681  0.670541 -0.012812            0.424139   
             EOS         -0.062072 -0.445252  0.123231            0.2

In [65]:
idx.value_counts()

False    1520
True      336
Name: count, dtype: int64

### Turn values into strings

In [66]:
unpivot = pivot.unstack().unstack().dropna().reset_index()  # values will be in col "0"
unpivot.loc[:, 1] = unpivot.loc[:, 0].astype("str")
unpivot.loc[:, 1] = unpivot.loc[:, 0].apply(lambda x: f"{round(x, ndigits=2):.2f}")
pivot = unpivot.pivot(index=["Measure Type", "Sim Meas."],
    columns=["Type", "Test", "Eval.", "Domain", "Dataset", "Arch."],
    values=1,)
pivot = pivot.reindex(measure_type_order, axis="index", level=0)  # <---

unpivot
pivot

Type                     Grounding by Prediction                          \
Test                                   Acc Corr.               JSD Corr.   
Eval.                                   Spearman                Spearman   
Domain                                     Graph                   Graph   
Dataset                                     Cora                    Cora   
Arch.                                        GAT    GCN   SAGE       GAT   
Measure Type Sim Meas.                                                     
CCA          PWCCA                         -0.26  -0.19   0.06     -0.06   
             SVCCA                         -0.16   0.07  -0.12      0.21   
Alignment    AlignCos                      -0.32   0.07   0.13      0.06   
             AngShape                      -0.13   0.07  -0.29      0.17   
             HardCorr                      -0.14   0.05  -0.11      0.39   
             LinReg                        -0.13   0.05  -0.22      0.06   
             OrthProc                      -0.13   0.07  -0.29      0.17   
             PermProc                      -0.28   0.11   0.19     -0.00   
             ProcDist                      -0.21   0.08   0.01     -0.06   
             SoftCorr                       0.02   0.19  -0.05      0.14   
RSM          CKA                            0.03   0.08  -0.18      0.49   
             DistCorr                       0.03   0.06  -0.18      0.53   
             EOS                           -0.05  -0.22   0.07     -0.06   
             GULP                          -0.12  -0.22   0.07     -0.09   
             RSA                           -0.31   0.15   0.04      0.09   
             RSMDiff                       -0.14  -0.05   0.08     -0.11   
Neighbors    2nd-Cos                       -0.12   0.11   0.04      0.42   
             Jaccard                       -0.15   0.16  -0.11      0.26   
             RankSim                        0.36   0.10  -0.12      0.52   
Topology     IMD                           -0.01  -0.08   0.01     -0.14   
Statistic    ConcDiff                      -0.22   0.18  -0.25      0.02   
             MagDiff                       -0.21   0.15  -0.13     -0.11   
             UnifDiff                      -0.08  -0.10  -0.05     -0.13   
Manifold     CKA_RBF                        0.09   0.07  -0.16      0.12   
             CKA_RBF_AUC                    0.02   0.06  -0.17      0.52   
             RWKA                          -0.03  -0.02  -0.16     -0.03   
             RWKA_AUC                      -0.21   0.07   0.01      0.10   

Type                                   Grounding by Design              \
Test                                         Random Labels               
Eval.                                                AUPRC               
Domain                                               Graph               
Dataset                                               Cora               
Arch.                       GCN   SAGE                 GAT   GCN  SAGE   
Measure Type Sim Meas.                                                   
CCA          PWCCA        -0.21   0.31                0.28  0.45  0.33   
             SVCCA         0.57   0.03                0.42  0.37  0.31   
Alignment    AlignCos      0.34   0.33                0.43  0.50  0.48   
             AngShape      0.68   0.21                0.42  0.43  0.42   
             HardCorr      0.67   0.13                0.42  0.42  0.42   
             LinReg        0.35   0.21                0.46  0.49  0.43   
             OrthProc      0.68   0.21                0.42  0.43  0.42   
             PermProc      0.67   0.21                0.44  0.45  0.39   
             ProcDist      0.65   0.16                0.43  0.46  0.45   
             SoftCorr      0.75   0.02                0.42  0.43  0.42   
RSM          CKA           0.67  -0.02                0.42  0.43  0.42   
             DistCorr      0.67  -0.01                0.42  0.43  0.44   
         

In [67]:
# Highlight the best values by bolding
for column in pivot.columns:
    col = pivot.loc[:, column].astype("float")
    idx = col == col.max()
    pivot.loc[idx, column] = pivot.loc[idx, column].apply(lambda s: r"\textbf{" + s + "}")
pivot

Type                     Grounding by Prediction                 \
Test                                   Acc Corr.                  
Eval.                                   Spearman                  
Domain                                     Graph                  
Dataset                                     Cora                  
Arch.                                        GAT            GCN   
Measure Type Sim Meas.                                            
CCA          PWCCA                         -0.26          -0.19   
             SVCCA                         -0.16           0.07   
Alignment    AlignCos                      -0.32           0.07   
             AngShape                      -0.13           0.07   
             HardCorr                      -0.14           0.05   
             LinReg                        -0.13           0.05   
             OrthProc                      -0.13           0.07   
             PermProc                      -0.28           0.11   
             ProcDist                      -0.21           0.08   
             SoftCorr                       0.02  \textbf{0.19}   
RSM          CKA                            0.03           0.08   
             DistCorr                       0.03           0.06   
             EOS                           -0.05          -0.22   
             GULP                          -0.12          -0.22   
             RSA                           -0.31           0.15   
             RSMDiff                       -0.14          -0.05   
Neighbors    2nd-Cos                       -0.12           0.11   
             Jaccard                       -0.15           0.16   
             RankSim               \textbf{0.36}           0.10   
Topology     IMD                           -0.01          -0.08   
Statistic    ConcDiff                      -0.22           0.18   
             MagDiff                       -0.21           0.15   
             UnifDiff                      -0.08          -0.10   
Manifold     CKA_RBF                        0.09           0.07   
             CKA_RBF_AUC                    0.02           0.06   
             RWKA                          -0.03          -0.02   
             RWKA_AUC                      -0.21           0.07   

Type                                                                   \
Test                                         JSD Corr.                  
Eval.                                         Spearman                  
Domain                                           Graph                  
Dataset                                           Cora                  
Arch.                              SAGE            GAT            GCN   
Measure Type Sim Meas.                                                  
CCA          PWCCA                 0.06          -0.06          -0.21   
             SVCCA                -0.12           0.21           0.57   
Alignment    AlignCos              0.13           0.06           0.34   
             AngShape             -0.29           0.17           0.68   
             HardCorr             -0.11           0.39           0.67   
             LinReg               -0.22           0.06           0.35   
             OrthProc             -0.29           0.17           0.68   
             PermProc     \textbf{0.19}          -0.00           0.67   
             ProcDist              0.01          -0.06           0.65   
             SoftCorr             -0.05           0.14           0.75   
RSM          CKA                  -0.18           0.49           0.67   
             DistCorr             -0.18  \textbf{0.53}           0.67   
             EOS                   0.07          -0.06          -0.45   
             GULP                  0.07          -0.09          -0.25   
             RSA                   0.04           0.09           0.49   
             RSMDiff               0.08          -0.11           0.35   
Neighbors    2nd-Cos               0.04   

### Significance Indicators

In [70]:
data_corr
os.makedirs(os.path.join(BASE_PATH, "results", "figs"), exist_ok=True)
data_corr.to_csv(os.path.join(BASE_PATH, "results", "figs", 
                              f"{target_dataset}_correlation_data.csv"))

In [69]:
# idx = data["Dataset"].isin(["SST2", "flickr", "IN100"]) & data["Arch."].isin(["SAGE", "BERT", "RNet18"]) & data.Test.isin(["Acc Corr.", "JSD Corr."])
# idx = data["Dataset"].isin(["MNLI", "flickr", "IN100"]) & data["Arch."].isin(["SAGE", "BERT", "RNet18"]) & data.Test.isin(["Acc Corr.", "JSD Corr."])
data_corr = data.copy()


def pval_str(pval):
    # if pval == pd.notna
    if isinstance(pval, float):
        if pval <= 0.01:
            return r"$^{**}$"
            # return r"$^{\dagger}$"
        if pval <= 0.05:
            return r"$^{*\phantom{*}}$"
            # return r"$^{\ddagger}$"
    return r"$^{\phantom{**}}$"

def significance_via_text_style(pval):
    if pval <= 0.01:
        return [r"\underline{\underline{", r"}}"]
    if pval <= 0.05:
        return [r"\underline{", r"}"]
    return ["", ""]

data_corr["val_comb"] = data_corr["value"].apply(lambda x: f"{round(x, ndigits=2):.2f}") + data_corr["pval"].apply(pval_str)
# data_corr["val_comb"] = data_corr["pval"].apply(significance_via_text_style).apply(lambda x: x[0]) + data_corr["value"].apply(lambda x: f"{round(x, ndigits=2):.2f}") + data_corr["pval"].apply(significance_via_text_style).apply(lambda x: x[1])
data_corr

pivot_corr = data_corr.pivot(
    index=["Measure Type", "Sim Meas."],
    columns=["Type", "Test", "Eval.", "Domain", "Dataset", "Arch."],
    values=["val_comb"],
).sort_values(
    by=["Measure Type", "Sim Meas."],
).reindex(
    measure_type_order, axis="index", level=0
).reindex(
    column_order, axis="columns", level="Test"
).reindex(
    ["Graph", "Text", "Vision"], axis="columns", level="Domain"
).loc[:, "val_comb"]
pivot_corr

def floatify(s: str) -> str:
    r"""Turn a string like '-0.10$^{\phantom{**}}$' into '-0.10'"""
    if not isinstance(s, str):
        return s
    return s[:s.find("$")]

def separate_significance_indicator(s: str) -> str:
    r"""Turn a string like '-0.10$^{\phantom{**}}$' into '$^{\phantom{**}}$'"""
    if not isinstance(s, str):
        return s
    return s[s.find("$"):]

for column in pivot_corr.columns:
    col = pivot_corr.loc[:, column].apply(floatify).astype("float")
    identifiers = pivot_corr.loc[:, column].apply(separate_significance_indicator)
    idx = col == col.max()
    new_col = col.apply(lambda x: f"{x:.2f}").apply(lambda s: r"\textbf{" + s + "}") + identifiers
    pivot_corr.loc[idx, column] = new_col

pivot_corr

ValueError: Index contains duplicate entries, cannot reshape

In [72]:
# pivot.loc[:, ("Grounding by Prediction")].astype("str", copy=False)
# pivot.loc[:, ("Grounding by Prediction", "Acc Corr.", "Spearman", "Graph", "flickr", "SAGE")] = pivot.loc[:, ("Grounding by Prediction", "Acc Corr.", "Spearman", "Graph", "flickr", "SAGE")].astype("str")
# pivot.loc[:, ("Grounding by Prediction")].dtypes

# pivot.loc[:, ("Grounding by Prediction")] = pivot_corr
pivot
os.makedirs(os.path.join(BASE_PATH, "results", "figs"), exist_ok=True)
pivot.to_csv(os.path.join(BASE_PATH, "results", "figs", 
                          f"{target_domain}_domain_table.csv"))

Step 3: Convert into latex table.

In [ ]:
styled = pd.io.formats.style.Styler(
    pivot,
    precision=2,
)

# Highlight top value
# latex_str = styled.highlight_max(axis=0, props="textbf:--rwrap;").to_latex(
#     hrules=True,
#     position="t",
#     label="tab:result_overview",
# )
latex_str = styled.to_latex(hrules=True, position="t", label="tab:result_overview",)


# ----- Manual modifications --------
latex_str = latex_str.split("\n")

# Center headers
pattern = r"\{r\}"
replacement = r"{c}"
latex_str = [re.sub(pattern, replacement, line) if i in [5, 6, 7] else line for i, line in enumerate(latex_str)]

# Remove measure row
latex_str.pop(11)

# Add vertical bars
line_no = 2
# line_no = 3
mod_line = latex_str[line_no][:18] + "".join(["|rrr"] * 6) + "}"
latex_str[line_no] = mod_line

# Make the left-most cells white
latex_str = [
    r"\cellcolor{white}" + line if i >= 11 and (i - 11) % 2 == 0 else line for i, line in enumerate(latex_str[:-4])
] + latex_str[-4:]
latex_str = "\n".join(latex_str)
print(latex_str)

\begin{table}[t]
\label{tab:result_overview}
\begin{tabular}{ll|rrr|rrr|rrr|rrr|rrr|rrr}
\toprule
 & Type & \multicolumn{6}{r}{Grounding by Prediction} & \multicolumn{18}{r}{Grounding by Design} \\
 & Test & \multicolumn{3}{c}{Acc Corr.} & \multicolumn{3}{c}{JSD Corr.} & \multicolumn{6}{c}{Random Labels} & \multicolumn{6}{c}{Shortcuts} & \multicolumn{6}{c}{Augmentation} \\
 & Eval. & \multicolumn{3}{c}{Spearman} & \multicolumn{3}{c}{Spearman} & \multicolumn{3}{c}{AUPRC} & \multicolumn{3}{c}{Conformity Rate} & \multicolumn{3}{c}{AUPRC} & \multicolumn{3}{c}{Conformity Rate} & \multicolumn{3}{c}{AUPRC} & \multicolumn{3}{c}{Conformity Rate} \\
 & Domain & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} & \multicolumn{3}{c}{Graph} \\
 & Dataset & \multicolumn{3}{r}{Cora} & \multicolumn{3}{r}{Cora} & \multicolumn{3}{r}{Cora} & \multicolumn{3}{r}{Co

## Rankplots

Requires section above to be run as well.

In [ ]:
import seaborn as sns

sns.set_theme("paper", style="white", font_scale=1.5)


Combine data similarly to before, but do not filter out specific parts.

In [ ]:
from IPython.display import display

In [76]:
# data = pd.concat([nlp_data, graph_data, vision_data])
data = graph_data.copy()
# display(data[(data.Setting == "mono") & (data.similarity_measure == "RTD")].head())
data = data.rename(
    columns={
        "functional_similarity_measure": "Functional Similarity Measure",
        "similarity_measure": "Representational Similarity Measure",
        "quality_measure": "Quality Measure",
    }
)
data = data.reset_index()


idx = data.Setting == "correlation"
data.loc[idx, "value"] = data.loc[idx, "corr"]

idx = data["Quality Measure"].isin(["AUPRC", "spearmanr", "correlation"])
data = data.loc[idx]

idx = data.Setting == "correlation"
data.loc[idx, "Setting"] = data.loc[idx, "Setting"] + data.loc[idx, "Functional Similarity Measure"]

idx = ~(data.Setting == "mono")
data.loc[idx, "model"] = "agg"

if target_domain == 'nlp':
    idx = data.Token.isna()
    data.loc[idx, "Token"] = "NA"

# idx = data.Token.isin(["mean"])
# data = data.loc[~idx]
if target_domain == 'nlp':
    data["rank"] = data.groupby(["domain", "Setting", "Dataset", "architecture", "model", "Token"], as_index=True)["value"].rank(
        ascending=False, method="min", na_option="keep"
    )
else:
    data["rank"] = data.groupby(["domain", "Setting", "Dataset", "architecture", "model"], as_index=True)["value"].rank(
        ascending=False, method="min", na_option="keep"
    )
display(data[(data.Setting == "mono") & (data["Representational Similarity Measure"] == "RTD")].head())


# combine layer mono results to equally weight experiments
idx = (data.model != "agg") & (~data["rank"].isna())
if target_domain == 'nlp':
    data.loc[idx, "rank"] = data[idx].groupby(["domain", "Setting", "Dataset", "architecture", "Token"])["rank"].transform("mean")
else:
    data.loc[idx, "rank"] = data[idx].groupby(["domain", "Setting", "Dataset", "architecture"])["rank"].transform("mean")
data = data.drop_duplicates(subset=["domain", "Setting", "Dataset", "architecture", "Representational Similarity Measure", "Functional Similarity Measure", "Quality Measure"])


data.loc[:, "Representational Similarity Measure"] = data["Representational Similarity Measure"].map(measure_to_abbrv)
data.loc[:, "architecture"] = data["architecture"].map(
    {
        "smollm2-1.7b": "SmolLM2",
        "albert-base-v2": "ALBERT",
        "BERT-L": "BERT",
        "GCN": "GCN",
        "GAT": "GAT",
        "GraphSAGE": "SAGE",
        "VGG11": "VGG11",
        "VGG19": "VGG19",
        "ResNet18": "RNet18",
        "ResNet34": "RNet34",
        "ResNet101": "RNet101",
        "ViT_B32": "ViT_B32",
        "ViT_L32": "ViT_L32",
    }
)
data.loc[:, "domain"] = data["domain"].map({"NLP": "Language", "GRAPHS": "Graph", "VISION": "Vision"})
data.loc[:, "Dataset"] = data["Dataset"].map(
    {
        "mnli_aug_rate0": "MNLI",
        "mnli_mem_rate0": "MNLI",
        "mnli": "MNLI",
        "sst2_sc_rate0558": "SST2",
        "sst2_mem_rate0": "SST2",
        "sst2_sft": "SST2",
        "sst2_sft_sc_rate0558": "SST2",
        "mnli_sc_rate0354": "MNLI",
        "sst2_aug_rate0": "SST2",
        "sst2": "SST2",
        "flickr": "flickr",
        "ogbn-arxiv": "arXiv",
        "cora": "Cora",
        "in100": "IN100",
        "c100": "CIFAR100",
    }
)
data.loc[:, "Setting"] = data["Setting"].map(
    {
        "aug": "Augmentation",
        "mem": "Random Labels",
        "correlationJSD": "JSD Corr.",
        "correlationAbsoluteAccDiff": "Acc Corr.",
        "correlationDisagreement": "Disagr. Corr.",
        "acc_corr": "Acc Corr.",
        "mono": "Layer Mono.",
        "sc": "Shortcuts",
    }
)
# display(data[(data.Setting == "Layer Mono.")  & (data["Representational Similarity Measure"] == "RTD")].head())


data.loc[:, "Quality Measure"] = data["Quality Measure"].map(
    {"violation_rate": "Conformity Rate", "AUPRC": "AUPRC", "spearmanr": "Spearman", "correlation": "Spearman"}
)
data.loc[data["Quality Measure"] == "Conformity Rate", "value"] = (
    1 - data.loc[data["Quality Measure"] == "Conformity Rate", "value"]
)  # must be run in conjunction with the above renaming
# display(data[(data.Setting == "Layer Mono.") & (data["Representational Similarity Measure"] == "RTD")].head())

data = data.rename(
    columns={
        "domain": "Modality",
        "architecture": "Arch.",
        "Representational Similarity Measure": "Sim Meas.",
        "Quality Measure": "Eval.",
        "Setting": "Scenario",
    }
)
# display(data[(data.Scenario == "Layer Mono.") & (data["Sim Meas."] == "RTD")].head())

data = data.sort_values(by=["Sim Meas."])

,index,Representational Similarity Measure,Quality Measure,value,domain,architecture,representation_dataset,identifier,Setting,Dataset,Functional Similarity Measure,corr,pval,model,rank


In [ ]:
fake = pd.DataFrame({"a": [2, 2, 2, 2, 2, 2, 3]})
fake.rank(method="min", ascending=False)

### Summary

Rank measures.

In [ ]:
avg_ranks = data.groupby(["Modality", "Sim Meas."])["rank"].agg(["mean", "median"]).reset_index()
avg_ranks = avg_ranks.rename(columns={"mean": "avg_rank", "median": "med_rank"})
avg_ranks

Create plots.

In [ ]:
plot_data = pd.merge(data, avg_ranks).sort_values(by=["med_rank", "avg_rank"])
plot_data = pd.merge(plot_data, pd.DataFrame.from_records(measure_types, columns=["Sim Meas.", "Measure Type"]), how="left", on="Sim Meas.")

fig, axes = plt.subplots(1, 3, sharey=False, figsize=(7*0.8*3, 7))
fig

for i, mod in enumerate(["Graph", "Language", "Vision"]):
    ax = axes[i]
    sns.boxplot(
        data=plot_data[plot_data.Modality == mod],
        x="rank",
        y="Sim Meas.",
        hue="Measure Type",
        hue_order=["Neighbors", "RSM", "Alignment", "Topology", "CCA", "Statistic"],
        palette="colorblind",
        legend=True if mod=="Vision" else False,
        ax=ax,
        # whis=(5.,95.)
    )
    ax.spines['right'].set_visible(False)
    ax.spines['top'].set_visible(False)

    ax.set_xlabel("Rank")
    ax.set_ylabel("Similarity Measures")

    fig.tight_layout()

    if mod == "Graph":
        ax.set_title("Graphs")
    else:
        ax.set_title(mod)

    if mod == "Vision":
        sns.move_legend(ax, loc="right", bbox_to_anchor=(1.45,0.5))
    fig.savefig(BASE_PATH / "figs" / f"aggregated_ver_{mod}.pdf", bbox_inches="tight")